In [ ]:
import cv2
import glob
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
pip install -U scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 3.3 MB/s eta 0:00:03
   ------ --------------------------------- 1.3/8.2 MB 3.5 MB/s eta 0:00:02
   ---------- ----------------------------- 2.1/8.2 MB 3.6 MB/s eta 0:00:02
   -------------- ------------------------- 2.9/8.2 MB 3.6 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.2 MB 3.6 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.2 MB 3.5 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.2 MB 3.6 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.2 MB 3.6 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.2 MB 3.6 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.2 MB 3.6 MB/s eta 0:00:01
   -------------------------------------- - 7.9/8.2 MB 3.6 MB/s eta 0:00:01
   --------------

In [5]:
pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import cv2
import numpy as np
import re

folder = "C:/Users/Vishnu/Downloads/CNN Project Images/CNN Project"

X = []
y = []

for file in os.listdir(folder):
    img = cv2.imread(os.path.join(folder, file))
    img = cv2.resize(img, (150,150))

    label = re.match(r"[A-Za-z]+", file).group()

    X.append(img)
    y.append(label)

X = np.array(X)
y = np.array(y)

In [3]:
import cv2
import numpy as np
import albumentations as A

transform = A.Compose([
    A.VerticalFlip(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.7),
    A.MedianBlur(p=0.7)   
])

In [4]:
X_aug = []
y_aug = []

for img, label in zip(X, y):
    for _ in range(20):   
        augmented = transform(image=img)
        X_aug.append(augmented["image"])
        y_aug.append(label)

X_aug = np.array(X_aug)
y_aug = np.array(y_aug)

print(X_aug.shape)
print(y_aug.shape)

(2080, 150, 150, 3)
(2080,)


In [5]:
X_final = np.concatenate((X, X_aug), axis=0)
y_final = np.concatenate((y, y_aug), axis=0)

print(X_final.shape)
print(y_final.shape)

(2184, 150, 150, 3)
(2184,)


In [6]:
label = LabelEncoder()
y_encoded = label.fit_transform(y_final)

print(label.classes_)   
print(y_encoded)  

['others' 'suzuki' 'toyota']
[0 0 0 ... 2 2 2]


In [7]:
X_final = X_final.astype("float32") / 255.0

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_final,y_encoded,test_size=0.2,random_state=42,)

In [9]:
model = Sequential([
    Conv2D(64, (3,3), activation='relu', input_shape=(150,150,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.25),
    Dense(3, activation='softmax')  
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

C:\Users\Vishnu\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.save("model.keras")

In [12]:
import joblib

joblib.dump(label, "label_encoder.pkl")

['label_encoder.pkl']